# Stage 1: Exploratory Data Analysis & Data Cleaning

**Objective**: Understand the LendingClub dataset before and after cleaning.

**Output**: `results/analysis_reports/` — pre-cleaning report, post-cleaning report, data card.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
sns.set_palette('muted')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12
})

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw' / 'wordsforthewise_lending-club'
REPORT_DIR = PROJECT_ROOT / 'results' / 'analysis_reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:     {RAW_DIR}')
print(f'Report dir:   {REPORT_DIR}')

In [ ]:
# Load raw data
df_raw = pd.read_csv(
    RAW_DIR / 'accepted_2007_to_2018Q4.csv.gz',
    compression='gzip',
    low_memory=False
)
print(f'Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

---
# PART 1: PRE-CLEANING ANALYSIS
---

## 1.1 Target Variable Analysis

In [ ]:
# Target distribution
target_counts = df_raw['loan_status'].value_counts()
target_pct = df_raw['loan_status'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
bars = ax.bar(target_counts.index, target_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Loan Status Distribution (Raw)')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
for bar, val, pct in zip(bars, target_counts.values, target_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
            f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=9)

ax = axes[1]
# Group: Good (Fully Paid), Bad (Charged Off+Default), Exclude (Current, Late, Grace, Policy)
good = target_counts.get('Fully Paid', 0)
bad = target_counts.get('Charged Off', 0) + target_counts.get('Default', 0)
exclude = target_counts.sum() - good - bad
cats = ['Good (Fully Paid)', 'Bad (Charged Off/Default)', 'Exclude (Current/Late/etc)']
vals = [good, bad, exclude]
colors = ['#2ca02c', '#d62728', '#7f7f7f']
wedges, texts, autotexts = ax.pie(vals, labels=cats, colors=colors, autopct='%1.1f%%',
                                   explode=(0, 0.05, 0.05), startangle=90)
ax.set_title('Good vs Bad vs Exclude')

plt.tight_layout()
plt.savefig(REPORT_DIR / '01_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Time-series analysis: default rate by year
df_raw['issue_year'] = pd.to_datetime(df_raw['issue_d'], format='%b-%Y', errors='coerce').dt.year

yearly = df_raw.groupby('issue_year').agg(
    total=('loan_status', 'count'),
    good=('loan_status', lambda x: (x == 'Fully Paid').sum()),
    bad=('loan_status', lambda x: (x.isin(['Charged Off', 'Default'])).sum()),
    current=('loan_status', lambda x: (x == 'Current').sum()),
).reset_index()
yearly['default_rate'] = yearly['bad'] / (yearly['good'] + yearly['bad']) * 100
yearly = yearly[yearly['issue_year'].between(2008, 2018)]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(yearly['issue_year'], yearly['default_rate'], 'o-', color='#d62728', linewidth=2, markersize=8)
ax.set_title('Default Rate by Loan Issue Year')
ax.set_xlabel('Year')
ax.set_ylabel('Default Rate (%)')
ax.set_xticks(yearly['issue_year'])
ax.grid(True, alpha=0.3)
for _, row in yearly.iterrows():
    ax.annotate(f"{row['default_rate']:.1f}%\n(n={row['total']:,})",
                (row['issue_year'], row['default_rate']),
                textcoords="offset points", xytext=(0, 15), fontsize=9, ha='center')
plt.tight_layout()
plt.savefig(REPORT_DIR / '02_default_rate_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.2 Missing Value Analysis

In [ ]:
# Overall missing rate distribution
missing_rates = df_raw.isnull().mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
ax = axes[0]
ax.hist(missing_rates * 100, bins=50, color='steelblue', edgecolor='white')
ax.axvline(x=80, color='#d62728', linestyle='--', label='Drop threshold (>80%)')
ax.axvline(x=20, color='#ff7f0e', linestyle='--', label='Flag threshold (20-80%)')
ax.set_title('Missing Rate Distribution Across 151 Features')
ax.set_xlabel('Missing Rate (%)')
ax.set_ylabel('Number of Features')
ax.legend()

# Categorize
drop_cols = (missing_rates > 0.8).sum()
flag_cols = ((missing_rates > 0.2) & (missing_rates <= 0.8)).sum()
keep_cols = (missing_rates <= 0.2).sum()

ax = axes[1]
ax.pie([drop_cols, flag_cols, keep_cols],
       labels=[f'Drop (>80%)\n{drop_cols} cols',
               f'Flag (20-80%)\n{flag_cols} cols',
               f'Keep (<20%)\n{keep_cols} cols'],
       colors=['#d62728', '#ff7f0e', '#2ca02c'],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Feature Missingness Categories')

plt.tight_layout()
plt.savefig(REPORT_DIR / '03_missing_rate_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Drop (>80% missing): {drop_cols} columns')
print(f'Flag (20-80% missing): {flag_cols} columns')
print(f'Keep (<20% missing): {keep_cols} columns')

## 1.3 Key Feature Distributions

In [ ]:
# Define the core group for pre-cleaning visualization
# (only loans with definite outcome)
mask_outcome = df_raw['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])
df_outcome = df_raw[mask_outcome].copy()
df_outcome['is_bad'] = df_outcome['loan_status'].isin(['Charged Off', 'Default']).astype(int)
print(f'Definite-outcome subset: {len(df_outcome):,} rows (bad rate: {df_outcome["is_bad"].mean():.2%})')

# Key features to plot
key_features = [
    ('loan_amnt', 'Loan Amount ($)'),
    ('int_rate', 'Interest Rate (%)'),
    ('annual_inc', 'Annual Income ($)'),
    ('dti', 'Debt-to-Income Ratio'),
    ('fico_range_low', 'FICO Score (Low)'),
    ('revol_util', 'Revolving Utilization (%)'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (col, label) in enumerate(key_features):
    ax = axes[i]
    good_data = df_outcome[df_outcome['is_bad'] == 0][col].dropna()
    bad_data = df_outcome[df_outcome['is_bad'] == 1][col].dropna()
    
    # Clip outliers for better visualization
    if col in ['annual_inc', 'dti', 'revol_util']:
        good_data = good_data.clip(upper=good_data.quantile(0.99))
        bad_data = bad_data.clip(upper=bad_data.quantile(0.99))
    
    ax.hist(good_data, bins=80, alpha=0.6, label='Fully Paid', color='#2ca02c', density=True)
    ax.hist(bad_data, bins=80, alpha=0.6, label='Charged Off', color='#d62728', density=True)
    ax.set_title(label)
    ax.legend()

axes[-1].set_visible(False)
plt.suptitle('Key Feature Distributions: Good vs Bad Loans (Pre-Cleaning)', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(REPORT_DIR / '04_feature_distributions_preclean.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.4 Categorical Feature Analysis

In [ ]:
# Default rate by grade, purpose, and home ownership
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, col in zip(axes, ['grade', 'purpose', 'home_ownership']):
    agg = df_outcome.groupby(col).agg(
        total=('is_bad', 'count'),
        bad_rate=('is_bad', 'mean')
    ).reset_index()
    agg = agg[agg['total'] > 100].sort_values('bad_rate', ascending=True)
    
    bars = ax.barh(agg[col], agg['bad_rate'] * 100, color='steelblue', edgecolor='white')
    ax.set_title(f'Default Rate by {col}')
    ax.set_xlabel('Default Rate (%)')
    for bar, val, total in zip(bars, agg['bad_rate'] * 100, agg['total']):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

plt.suptitle('Categorical Feature Risk Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(REPORT_DIR / '05_categorical_risk_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 Correlation Structure

In [ ]:
# Select numeric columns with <20% missing for correlation
numeric_cols = df_outcome.select_dtypes(include=[np.number]).columns
low_missing_num = [c for c in numeric_cols if df_outcome[c].isnull().mean() < 0.2]

# Pick top ~25 features by variance
variances = df_outcome[low_missing_num].var().sort_values(ascending=False)
top_features = variances.head(25).index.tolist()

corr = df_outcome[top_features].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'fontsize': 7})
ax.set_title('Feature Correlation Matrix (Top 25 Numeric Features)', fontsize=16)
plt.tight_layout()
plt.savefig(REPORT_DIR / '06_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.6 Data Leakage Audit

In [ ]:
# List features that are only observable AFTER loan issuance
leaky_features = [
    'out_prncp', 'out_prncp_inv',           # outstanding principal
    'total_pymnt', 'total_pymnt_inv',       # total payment received
    'total_rec_prncp', 'total_rec_int',     # principal/interest received
    'total_rec_late_fee',                   # late fees received
    'recoveries', 'collection_recovery_fee', # recovery amounts
    'last_pymnt_d', 'last_pymnt_amnt',      # last payment info
    'next_pymnt_d',                         # next payment date
    'last_credit_pull_d',                   # last credit pull (may be post-issuance)
    'debt_settlement_flag',                 # settlement flag
    'settlement_status', 'settlement_date',
    'settlement_amount', 'settlement_percentage', 'settlement_term',
    'hardship_flag', 'hardship_type', 'hardship_reason',
    'hardship_status', 'hardship_start_date', 'hardship_end_date',
    'hardship_amount', 'hardship_length', 'hardship_dpd',
    'deferral_term', 'payment_plan_start_date',
    'orig_projected_additional_accrued_interest',
]

existing_leaky = [c for c in leaky_features if c in df_raw.columns]
print(f'Leaky features to remove: {len(existing_leaky)}')
for c in existing_leaky:
    print(f'  - {c}')

## 1.7 Pre-Cleaning Summary

In [ ]:
preclean_summary = f"""
========================================
  PRE-CLEANING DATA SUMMARY
========================================
Rows:               {df_raw.shape[0]:,}
Columns:            {df_raw.shape[1]}
Date range:         {df_raw['issue_d'].dropna().min()} -> {df_raw['issue_d'].dropna().max()}
Target categories:  {df_raw['loan_status'].nunique()}

Target breakdown:
  Fully Paid:       {target_counts.get('Fully Paid', 0):,} ({target_pct.get('Fully Paid', 0):.1f}%)
  Charged Off:      {target_counts.get('Charged Off', 0):,} ({target_pct.get('Charged Off', 0):.1f}%)
  Current:          {target_counts.get('Current', 0):,} ({target_pct.get('Current', 0):.1f}%)
  Late (31-120):    {target_counts.get('Late (31-120 days)', 0):,} ({target_pct.get('Late (31-120 days)', 0):.1f}%)
  Other:            {target_counts.sum() - target_counts.get('Fully Paid',0) - target_counts.get('Charged Off',0) - target_counts.get('Current',0) - target_counts.get('Late (31-120 days)',0):,}

Missingness:
  Drop (>80%):      {drop_cols} columns
  Flag (20-80%):    {flag_cols} columns
  Keep (<20%):      {keep_cols} columns

Leaky features:     {len(existing_leaky)} identified

Key risks:
  - 38.9% of loans are 'Current' (no final outcome), must be excluded
  - ~36 hardship/settlement features are near-100% missing or post-issuance
  - Joint-application features (annual_inc_joint, etc.) ~95% missing
  - member_id is 100% missing (useless)
  - desc (loan description text) 94% missing
  - url is a data artifact, not a feature
========================================
"""

print(preclean_summary)
with open(REPORT_DIR / 'pre_cleaning_summary.txt', 'w') as f:
    f.write(preclean_summary)

---
# PART 2: DATA CLEANING
---

In [ ]:
from src.data.cleaning import (
    define_labels,
    drop_high_missingness,
    drop_leaky_features,
    drop_id_artifact_columns,
    apply_temporal_split,
    CLEANING_STATS,
)

print('Cleaning functions imported from src.data.cleaning')

In [ ]:
# Execute full cleaning pipeline
df = df_raw.copy()

# Step 1: Define target labels (exclude Current/Late/Grace)
df = define_labels(df)

# Step 2: Drop ID/artifact columns
df = drop_id_artifact_columns(df)

# Step 3: Drop near-100% missing columns (>80%)
df = drop_high_missingness(df, threshold=0.8)

# Step 4: Drop leaky (post-issuance) features
df = drop_leaky_features(df)

# Step 5: Temporal split (train 2014-2016, val 2017, test 2018)
train, val, test = apply_temporal_split(df)

---
# PART 3: POST-CLEANING ANALYSIS
---

In [ ]:
print(f'Post-cleaning shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')
print(f'Train bad rate: {train["is_bad"].mean():.2%}')
print(f'Val   bad rate: {val["is_bad"].mean():.2%}')
print(f'Test  bad rate: {test["is_bad"].mean():.2%}')

## 3.1 Post-Cleaning Feature Distributions

In [ ]:
# Compare key features before vs after cleaning (with train-test overlap)
key_features = [
    ('loan_amnt', 'Loan Amount ($)'),
    ('int_rate', 'Interest Rate (%)'),
    ('annual_inc', 'Annual Income ($)'),
    ('dti', 'Debt-to-Income Ratio'),
    ('fico_range_low', 'FICO Score (Low)'),
    ('revol_util', 'Revolving Utilization (%)'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (col, label) in enumerate(key_features):
    ax = axes[i]
    if col not in train.columns:
        continue
    
    good_data = train[train['is_bad'] == 0][col].dropna()
    bad_data = train[train['is_bad'] == 1][col].dropna()
    
    if col in ['annual_inc', 'dti', 'revol_util']:
        good_data = good_data.clip(upper=good_data.quantile(0.99))
        bad_data = bad_data.clip(upper=bad_data.quantile(0.99))
    
    ax.hist(good_data, bins=80, alpha=0.6, label='Fully Paid', color='#2ca02c', density=True)
    ax.hist(bad_data, bins=80, alpha=0.6, label='Charged Off', color='#d62728', density=True)
    ax.set_title(f'{label} (Post-Cleaning, Train Set)')
    ax.legend()

axes[-1].set_visible(False)
plt.suptitle('Key Feature Distributions: Good vs Bad Loans (Post-Cleaning)', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(REPORT_DIR / '07_feature_distributions_postclean.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.2 Train/Val/Test Distribution Check

In [ ]:
# Check that train/val/test have similar distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, (col, label) in enumerate(key_features):
    ax = axes[i]
    if col not in train.columns:
        continue
    
    for split_name, split_df in [('Train', train), ('Val', val), ('Test', test)]:
        data = split_df[col].dropna()
        if col in ['annual_inc', 'dti', 'revol_util']:
            data = data.clip(upper=data.quantile(0.99))
        ax.hist(data, bins=60, alpha=0.4, label=split_name, density=True)
    
    ax.set_title(f'{label} — Train/Val/Test')
    ax.legend()

axes[-1].set_visible(False)
plt.suptitle('Train / Val / Test Distribution Alignment', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(REPORT_DIR / '08_train_val_test_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Post-Cleaning Correlation Analysis

In [ ]:
# Post-cleaning correlation (training set)
numeric_cols_clean = train.select_dtypes(include=[np.number]).columns
low_missing_num_clean = [c for c in numeric_cols_clean if train[c].isnull().mean() < 0.2]
variances_clean = train[low_missing_num_clean].var().sort_values(ascending=False)
top_features_clean = variances_clean.head(25).index.tolist()

# Ensure is_bad is included
if 'is_bad' not in top_features_clean:
    top_features_clean = ['is_bad'] + top_features_clean[:24]

corr_clean = train[top_features_clean].corr()

fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_clean, dtype=bool), k=1)
sns.heatmap(corr_clean, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'fontsize': 7})
ax.set_title('Post-Cleaning Correlation Matrix (Train Set)', fontsize=16)
plt.tight_layout()
plt.savefig(REPORT_DIR / '09_correlation_postclean.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.4 Feature-Target Association (Top Predictors)

In [ ]:
# Compute point-biserial correlation or AUC for each numeric feature vs target
from sklearn.metrics import roc_auc_score

numeric_features = [c for c in train.select_dtypes(include=[np.number]).columns
                    if c != 'is_bad' and train[c].notna().sum() > 1000]

feature_auc = {}
for col in numeric_features:
    valid = train[col].notna() & train['is_bad'].notna()
    if valid.sum() < 1000:
        continue
    try:
        auc = roc_auc_score(train.loc[valid, 'is_bad'], train.loc[valid, col])
        feature_auc[col] = max(auc, 1 - auc)  # make sure direction doesn't matter
    except Exception:
        pass

top_predictors = sorted(feature_auc.items(), key=lambda x: x[1], reverse=True)[:20]

fig, ax = plt.subplots(figsize=(10, 8))
cols, aucs = zip(*top_predictors)
bars = ax.barh(range(len(cols)), aucs, color='steelblue', edgecolor='white')
ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols)
ax.invert_yaxis()
ax.set_xlabel('Univariate AUC')
ax.set_title('Top 20 Predictive Features (Univariate AUC)')
for i, (col, auc) in enumerate(zip(cols, aucs)):
    ax.text(auc + 0.005, i, f'{auc:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(REPORT_DIR / '10_top_predictors.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.5 Post-Cleaning Summary & Data Card

In [ ]:
postclean_summary = f"""
========================================
  POST-CLEANING DATA SUMMARY
========================================
Cleaned rows:       {df.shape[0]:,}
Cleaned columns:    {df.shape[1]}

Temporal split:
  Train (2014-2016):  {len(train):,} ({len(train)/len(df)*100:.1f}%)  bad_rate={train['is_bad'].mean():.2%}
  Val (2017):         {len(val):,} ({len(val)/len(df)*100:.1f}%)  bad_rate={val['is_bad'].mean():.2%}
  Test (2018):        {len(test):,} ({len(test)/len(df)*100:.1f}%)  bad_rate={test['is_bad'].mean():.2%}

Features removed:   {CLEANING_STATS.get('total_removed', 'N/A')}
Features kept:      {df.shape[1]}

Cleaning steps applied:
  1. Label definition: 'Charged Off'+'Default' = bad, 'Fully Paid' = good
  2. Excluded: 'Current', 'Late', 'In Grace Period', policy-violation rows
  3. Dropped >80% missing columns
  4. Dropped post-issuance (leaky) features
  5. Dropped ID/artifact columns (id, member_id, url, desc, etc.)
  6. Temporal split: train=2014-2016, val=2017, test=2018

Remaining risks:
  - Some features still have moderate missingness (flag category)
  - emp_title is free-text, needs NLP/embedding or exclusion
  - zip_code and addr_state are location proxies (fairness concern)
  - Survivorship bias: training data only contains approved loans
========================================
"""

print(postclean_summary)
with open(REPORT_DIR / 'post_cleaning_summary.txt', 'w') as f:
    f.write(postclean_summary)
print(f'\nReports saved to: {REPORT_DIR}')